# Using the PhenoMe pipeline with STED-FM models and data
## Neural Activity States (PSD-95)
---

## 1. Setup

In [1]:
import torch
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

### Wrap the STED-FM model

In [2]:
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

wrapper = STEDFMWrapper(device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### Using DINOv2

In [14]:
wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


## Synaptic Proteins Datasets

### Zenodo segmentation dataset

In [4]:
from data_handling import SemanticProteinSegmentationDataset
import h5py
from matplotlib import pyplot as plt

h5file = "/Users/renaud/datasets/synaptic_proteins/test_2024-05-16.hdf5"

dataset = SemanticProteinSegmentationDataset(
    h5file=h5file,
    n_channels=1,
    data_aug=0,
    validation=True,
    image_size=224,
    return_metadata=True,  # __getitem__ also yields the protein/condition of each tile
)


100%|██████████| 1357/1357 [00:00<00:00, 3032.03it/s]


In [ ]:
# Export the dataset tiles as they come out of __getitem__: the image in tiles_images_dir,
# its soft label stack (one channel per synapse class) in tiles_masks_dir, and one metadata
# row per tile in metadata.csv. Image, mask and metadata row share the same filename stem.
import os

import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

tiles_dir = "/Users/renaud/datasets/synaptic_proteins/tiles_224"
tiles_images_dir = os.path.join(tiles_dir, "images")
tiles_masks_dir = os.path.join(tiles_dir, "masks")
tiles_metadata_path = os.path.join(tiles_dir, "metadata.csv")

os.makedirs(tiles_images_dir, exist_ok=True)
os.makedirs(tiles_masks_dir, exist_ok=True)

rows = []
for idx in tqdm(range(len(dataset)), desc="Saving tiles"):
    # Requires return_metadata=True above; metadata carries the protein, condition and
    # the tile origin (channel, top-left corner) in the original image.
    image, mask, metadata = dataset[idx]

    stem = (
        f"{os.path.splitext(metadata['image_name'])[0]}"
        f"_ch{metadata['channel']}_r{metadata['row']:04d}_c{metadata['col']:04d}"
    )

    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 3 and image.shape[0] == 1:
        image = image[0]  # (1, H, W) -> (H, W)
    mask = np.asarray(mask, dtype=np.float32)  # (n_classes, H, W)

    tifffile.imwrite(os.path.join(tiles_images_dir, f"{stem}.tif"), image)
    tifffile.imwrite(os.path.join(tiles_masks_dir, f"{stem}.tif"), mask)
    rows.append({"filename": stem, **metadata})

# 'filename' is the extension-less stem shared by the image and its mask, which is what
# make_dataframe_metadata_fn matches on in pheno.
# 
# ().
tiles_metadata_df = pd.DataFrame(rows)
tiles_metadata_df.to_csv(tiles_metadata_path, index=False)

print(f"Saved {len(tiles_metadata_df)} tiles to {tiles_images_dir} and {tiles_masks_dir}")
print(f"Metadata written to {tiles_metadata_path}")
tiles_metadata_df.head()

### Patch Retrieval Dataset (from STED-FM paper)

In [5]:
from stedfm.datasets import NeuralActivityStates

# Instantiate the dataset object
dataset = NeuralActivityStates(tarpath="/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2.tar", balance=False)


Processing dataset..: 100%|██████████| 1510/1510 [00:00<00:00, 5819.91it/s]


In [ ]:
# Iterate through dataset and save individual images, masks and metadata.csv
import os

import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

tiles_dir = "/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2"
tiles_images_dir = os.path.join(tiles_dir, "images")
tiles_masks_dir = os.path.join(tiles_dir, "masks")
tiles_metadata_path = os.path.join(tiles_dir, "metadata.csv")

os.makedirs(tiles_images_dir, exist_ok=True)
os.makedirs(tiles_masks_dir, exist_ok=True)

rows = []
for image, meta in tqdm(dataset, desc="Saving tiles"):

    stem = (
        f"img_{meta.get('dataset-idx')}"
    )

    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 3 and image.shape[0] == 1:
        image = image[0]  # (1, H, W) -> (H, W)
    mask = np.asarray(meta.get('mask'), dtype=np.float32)  # (n_classes, H, W)

    tifffile.imwrite(os.path.join(tiles_images_dir, f"{stem}.tif"), image)
    tifffile.imwrite(os.path.join(tiles_masks_dir, f"{stem}.tif"), mask)
    row_dict = {k: meta.get(k) for k in meta.keys() if k != 'mask'}
    row_dict["filename"] = stem
    rows.append(row_dict)

# 'filename' is the extension-less stem shared by the image and its mask, which is what
# make_dataframe_metadata_fn matches on in pheno.find_files().
tiles_metadata_df = pd.DataFrame(rows)
tiles_metadata_df.to_csv(tiles_metadata_path, index=False)

print(f"Saved {len(tiles_metadata_df)} tiles to {tiles_images_dir} and {tiles_masks_dir}")
print(f"Metadata written to {tiles_metadata_path}")
tiles_metadata_df.head()

Saving tiles: 100%|██████████| 1510/1510 [00:00<00:00, 2195.43it/s]

Saved 1510 tiles to /Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/images and /Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/masks
Metadata written to /Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/metadata.csv


,label,condition,dataset-idx,filename
0,1,0MgGlyBic,0,img_0
1,1,0MgGlyBic,1,img_1
2,1,0MgGlyBic,2,img_2
3,1,0MgGlyBic,3,img_3
4,1,0MgGlyBic,4,img_4


### Custom transforms for STED-FM model

In [3]:
import torchvision.transforms as T

# Transforms for STED-FM model; normalisation is applied in the data loader
stedfm_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

### Create dataset (find files)

In [4]:
# csv-based metadata
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

# Segmentation dataset
#images_dir = "/Users/renaud/datasets/synaptic_proteins/tiles_224/images"
#masks_dir = None
#metadata_dir = "/Users/renaud/datasets/synaptic_proteins/tiles_224/metadata.csv"

# PSD95 dataset
images_dir = "/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/images"
masks_dir = None
metadata_dir = "/Users/renaud/datasets/synaptic_proteins/NAS_PSD95_test_v2/metadata.csv"

if metadata_dir:
    metadata_df = pd.read_csv(metadata_dir)
    metadata_fn = make_dataframe_metadata_fn(metadata_df)
else:
    metadata_fn = None
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")



Found 1510 files total.
DataFrame: 1510 files, 7 columns.


Found 1510 images
Metadata keys: ['file_path', 'label', 'condition', 'dataset-idx', 'filename', 'id', 'mask_path']


### Process images

In [5]:
# Process with STED-FM model
checkpoint_path = "checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5"


pheno.process_images(wrapper, 
                    force_rgb=False,
                    checkpoint_path=checkpoint_path,
                    custom_transformations=stedfm_transforms,
                    #preprocessing_fn=minmax_preprocessing_fn
                    )
pheno.compute_properties(property_preset="complete",checkpoint_path=checkpoint_path)

Found 1510 committed images; skipping 1510 (already in checkpoint), 0 to process.
All images already processed. Loading from checkpoint.
Pipeline ready: 1510 images available (lazy loading from checkpoints/psd95_dataset_whole/psd95_dataset_stedfm.h5).
Properties: 1510/1510 matched in checkpoint (6 columns).


,image_index,image_path,image_name,intensity_entropy,intensity_max,intensity_mean,intensity_min,intensity_std,sharpness_metric
0,0,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_0.tif,1.992795,0.367381,0.006625,0.0,0.029033,0.441279
1,1,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_1.tif,3.593781,0.286102,0.008940,0.0,0.020412,0.343669
2,2,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_10.tif,2.275711,0.206449,0.004254,0.0,0.014439,0.356436
3,3,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_100.tif,5.073238,0.145820,0.011388,0.0,0.016860,0.406070
4,4,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_1000.tif,3.144090,0.537190,0.027829,0.0,0.054342,0.355968
...,...,...,...,...,...,...,...,...,...
1505,1505,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_995.tif,2.167955,1.000000,0.021172,0.0,0.087714,0.398192
1506,1506,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_996.tif,3.713688,0.727273,0.043963,0.0,0.079111,0.387257
1507,1507,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_997.tif,3.182842,0.702479,0.031839,0.0,0.067542,0.382085
1508,1508,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_998.tif,2.714777,0.636364,0.022045,0.0,0.052579,0.335323


In [25]:
# Process with DINO-v2 model
checkpoint_path="checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5"


pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=None, # Default DinoV2 transforms
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/48 [00:00<?, ?it/s]

Pipeline ready: 1510 images available (lazy loading from checkpoints/psd95_dataset_whole/psd95_dataset_dinov2.h5).


Computing properties:   0%|          | 0/1510 [00:00<?, ?img/s]

,image_index,image_path,image_name,intensity_std,intensity_mean
0,0,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_0.tif,0.029033,0.006625
1,1,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_1.tif,0.020412,0.008940
2,2,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_10.tif,0.014439,0.004254
3,3,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_100.tif,0.016860,0.011388
4,4,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_1000.tif,0.054342,0.027829
...,...,...,...,...,...
1505,1505,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_995.tif,0.087714,0.021172
1506,1506,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_996.tif,0.079111,0.043963
1507,1507,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_997.tif,0.067542,0.031839
1508,1508,/Users/renaud/datasets/synaptic_proteins/NAS_P...,img_998.tif,0.052579,0.022045


In [9]:
# Plot number of images per group
pheno.plot_counts(group_by=["condition"])

## Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [8]:
pheno.compute_clustering(n_clusters=4, clustering_method="kmeans", normalize=True)
print(pheno.get_available_property_keys())
print(pheno.get_available_metadata_keys())

pheno.plot_tsne(color_by="condition")
pheno.plot_tsne(color_by="cluster")

Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 4 clusters on 1510 images (source=embeddings).


['intensity_entropy', 'intensity_max', 'intensity_mean', 'intensity_min', 'intensity_std', 'sharpness_metric']
['cluster', 'condition', 'dataset-idx', 'filename', 'id', 'label', 'mask_path']


## Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [8]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

<IPython.core.display.Javascript object>